# Project 2 - Cell / Particle Counter

This notebook demonstrates a classical watershed-based segmentation pipeline using the exact same processing modules as the desktop application.

## 1. Objective

Segment touching cells or particles, count them, and analyze how parameter changes affect the result.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from apps.cell_counter.processing.distance_transform import compute_distance_transform
from apps.cell_counter.processing.morphology import clean_binary_mask
from apps.cell_counter.processing.preprocessing import gaussian_blur, to_grayscale
from apps.cell_counter.processing.threshold import apply_threshold
from apps.cell_counter.processing.watershed import CellCounterParams, run_watershed_pipeline
from shared.image_io import list_images, load_image
from shared.image_utils import bgr_to_rgb

DATASET = Path('datasets/project_2/input')
image_paths = list_images(DATASET)
print(f'Found {len(image_paths)} images in {DATASET}')
image_paths[:5]

## 2. Dataset

Place at least 10 cell or particle images inside `datasets/project_2/input/` before executing the notebook.

In [ ]:
if not image_paths:
    raise SystemExit('Please place sample cell images into datasets/project_2/input before running the notebook.')

sample_image = load_image(image_paths[0])
plt.figure(figsize=(6, 6))
plt.imshow(bgr_to_rgb(sample_image))
plt.title(f'Sample image: {image_paths[0].name}')
plt.axis('off');

## 3. Preprocessing

In [ ]:
gray = to_grayscale(sample_image)
blurred = gaussian_blur(gray, 5, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(bgr_to_rgb(sample_image))
axes[0].set_title('Original')
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Grayscale')
axes[2].imshow(blurred, cmap='gray')
axes[2].set_title('Gaussian Blur')
for axis in axes:
    axis.axis('off')
plt.tight_layout();

## 4. Thresholding

In [ ]:
binary_manual, _ = apply_threshold(blurred, 'Binary', 127)
binary_otsu, otsu_value = apply_threshold(blurred, 'Otsu')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(binary_manual, cmap='gray')
axes[0].set_title('Manual threshold')
axes[1].imshow(binary_otsu, cmap='gray')
axes[1].set_title(f'Otsu threshold (value={otsu_value:.1f})')
for axis in axes:
    axis.axis('off')
plt.tight_layout();

## 5. Morphological Operations

In [ ]:
opened, sure_background = clean_binary_mask(binary_otsu, 3, 2, 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(opened, cmap='gray')
axes[0].set_title('Morphology / Opening')
axes[1].imshow(sure_background, cmap='gray')
axes[1].set_title('Sure Background')
for axis in axes:
    axis.axis('off')
plt.tight_layout();

## 6. Distance Transform

In [ ]:
distance_result = compute_distance_transform(opened, 'L2', 5, 0.4)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(distance_result.distance_map, cmap='magma')
axes[0].set_title('Raw distance map')
axes[1].imshow(distance_result.normalized_map, cmap='magma')
axes[1].set_title('Normalized distance map')
axes[2].imshow(distance_result.sure_foreground, cmap='gray')
axes[2].set_title('Sure foreground')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
print('Max distance:', distance_result.max_distance)
print('Threshold value:', distance_result.threshold_value)

## 7. Marker Generation and Watershed

In [ ]:
params = CellCounterParams(distance_threshold_ratio=0.4)
result = run_watershed_pipeline(sample_image, params)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(result.sure_foreground, cmap='gray')
axes[0].set_title('Sure foreground')
axes[1].imshow(result.unknown_region, cmap='gray')
axes[1].set_title('Unknown region')
axes[2].imshow(result.marker_visualization)
axes[2].set_title('Markers / Watershed')
axes[3].imshow(bgr_to_rgb(result.overlay))
axes[3].set_title(f'Final overlay ({result.object_count} objects)')
for axis in axes:
    axis.axis('off')
plt.tight_layout();

## 8. Parameter Experiment

In [ ]:
rows = []
for threshold_ratio in [0.20, 0.30, 0.40, 0.50, 0.60]:
    experiment_result = run_watershed_pipeline(sample_image, CellCounterParams(distance_threshold_ratio=threshold_ratio))
    rows.append({
        'Threshold': threshold_ratio,
        'Detected Objects': experiment_result.object_count,
        'Average Area': experiment_result.average_area,
        'Processing Time (ms)': experiment_result.processing_time_ms,
    })

display(HTML('<table border="1"><tr><th>Threshold</th><th>Detected Objects</th><th>Average Area</th><th>Processing Time (ms)</th></tr>' + ''.join(
    f"<tr><td>{row['Threshold']:.2f}</td><td>{row['Detected Objects']}</td><td>{row['Average Area']:.2f}</td><td>{row['Processing Time (ms)']:.2f}</td></tr>" for row in rows
) + '</table>'))

In [ ]:
thresholds = [row['Threshold'] for row in rows]
counts = [row['Detected Objects'] for row in rows]
plt.figure(figsize=(6, 4))
plt.plot(thresholds, counts, marker='o')
plt.title('Threshold vs Object Count')
plt.xlabel('Threshold')
plt.ylabel('Detected Objects')
plt.grid(True)

## 9. Test on Multiple Images

In [ ]:
rows = []
for path in image_paths[:10]:
    image = load_image(path)
    pipeline_result = run_watershed_pipeline(image, CellCounterParams())
    rows.append({
        'Image': path.name,
        'Detected objects': pipeline_result.object_count,
        'Processing time (ms)': pipeline_result.processing_time_ms,
        'Average area': pipeline_result.average_area,
    })

display(HTML('<table border="1"><tr><th>Image</th><th>Detected objects</th><th>Processing time (ms)</th><th>Average area</th></tr>' + ''.join(
    f"<tr><td>{row['Image']}</td><td>{row['Detected objects']}</td><td>{row['Processing time (ms)']:.2f}</td><td>{row['Average area']:.2f}</td></tr>" for row in rows
) + '</table>'))

## 10. Limitations

- Strong overlap can merge objects.
- Poor contrast may weaken thresholding.
- Very small noise particles may create extra markers.
- The pipeline is sensitive to distance-threshold and morphology settings.

## 11. Conclusion

The watershed pipeline provides an explainable classical computer vision solution for segmenting and counting cells while exposing each processing stage for analysis.